# Assignment 23: OpenAI and Retrieval Augmented Generation (RAG)

In [10]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [11]:
load_dotenv()

True

# Part 1: OpenAI Setup and Basic Prompt

## TASK 1: OpenAI Setup and Basic Prompt

In [12]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [13]:
prompt = "What is Generative AI? Explain in 3 simple points."

In [14]:
response = llm.invoke(prompt)

In [15]:
response.content

'Generative AI refers to a type of artificial intelligence that can create new content or data. Here are three simple points to explain it:\n\n1. **Content Creation**: Generative AI can produce various forms of content, such as text, images, music, and videos, by learning patterns from existing data.\n\n2. **Learning from Data**: It uses algorithms, particularly deep learning models, to analyze large datasets and understand the underlying structures, enabling it to generate new, similar content.\n\n3. **Applications**: Generative AI is used in various fields, including art, writing, gaming, and even drug discovery, helping to automate creative processes and enhance productivity.'

# Part 2: Retriever-Based RAG (Wikipedia + Vector Store)

## Task 2: Wikipedia Retriever

In [16]:
from langchain_community.retrievers import WikipediaRetriever

In [17]:
retriever = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=3000
)

In [18]:
query = "Artificial Intelligence"

In [22]:
documents = retriever.invoke({query: query})

TypeError: unhashable type: 'slice'

In [20]:
documents

[Document(metadata={}, page_content='\n        Artificial Intelligence is a field of computer science\n        that focuses on creating systems capable of performing\n        tasks that normally require human intelligence.\n        '),
 Document(metadata={}, page_content='\n        Machine Learning is a subset of Artificial Intelligence.\n        It allows computers to learn patterns from data and make\n        predictions or decisions without being explicitly programmed.\n        '),
 Document(metadata={}, page_content='\n        Deep Learning is a subset of Machine Learning that uses\n        neural networks with multiple layers to learn complex\n        patterns from large datasets.\n        '),
 Document(metadata={}, page_content='\n        Generative AI is artificial intelligence that can generate\n        new content such as text, images, audio, video, and code.\n        ')]

In [21]:
for i, doc in enumerate(documents, start=1):
    print("Title:", doc.metadata.get("title"))
    print(doc.page_content)

Title: None

        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        
Title: None

        Machine Learning is a subset of Artificial Intelligence.
        It allows computers to learn patterns from data and make
        predictions or decisions without being explicitly programmed.
        
Title: None

        Deep Learning is a subset of Machine Learning that uses
        neural networks with multiple layers to learn complex
        patterns from large datasets.
        
Title: None

        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        


## Task 3: Vector Store Retriever

In [24]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [25]:
documents = [
    Document(
        page_content="""
        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        """
    ),
    Document(
        page_content="""
        Machine Learning is a subset of Artificial Intelligence.
        It allows computers to learn patterns from data and make
        predictions or decisions without being explicitly programmed.
        """
    ),
    Document(
        page_content="""
        Deep Learning is a subset of Machine Learning that uses
        neural networks with multiple layers to learn complex
        patterns from large datasets.
        """
    ),
    Document(
        page_content="""
        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        """
    ),
]

In [26]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [27]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [28]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [29]:

query = "What is Generative AI?"

results = retriever.invoke(query)

In [30]:
for i, doc in enumerate(results, start=1):
    print(f"\nDocument {i}:")
    print(doc.page_content)


Document 1:

        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        

Document 2:

        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        


# Part 3: Advanced Retrieval Strategies

## Task 4: Maximal Marginal Relevance ( MMR ) Retriever

In [31]:
query = "What are the applications of artificial intelligence?"

In [32]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

similarity_results = similarity_retriever.invoke(query)


print("\nNORMAL SIMILARITY SEARCH")
print("=" * 60)

for i, doc in enumerate(similarity_results, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)


NORMAL SIMILARITY SEARCH

Document 1:

        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        

Document 2:

        Machine Learning is a subset of Artificial Intelligence.
        It allows computers to learn patterns from data and make
        predictions or decisions without being explicitly programmed.
        

Document 3:

        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        


In [33]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 5,
        "lambda_mult": 0.5
    }
)

mmr_results = mmr_retriever.invoke(query)

In [34]:
for i, doc in enumerate(mmr_results, 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)


Document 1:

        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        

Document 2:

        Deep Learning is a subset of Machine Learning that uses
        neural networks with multiple layers to learn complex
        patterns from large datasets.
        

Document 3:

        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        


## TASk 5: Multi Query Retriever

In [36]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [38]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [39]:

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [40]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [41]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [42]:
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

In [43]:

query = "How does AI help healthcare?"

In [45]:

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):

    print(f"\nDocument {i}:")
    print(doc.page_content)


Document 1:

        Artificial Intelligence is a field of computer science
        that focuses on creating systems capable of performing
        tasks that normally require human intelligence.
        

Document 2:

        Machine Learning is a subset of Artificial Intelligence.
        It allows computers to learn patterns from data and make
        predictions or decisions without being explicitly programmed.
        

Document 3:

        Generative AI is artificial intelligence that can generate
        new content such as text, images, audio, video, and code.
        


## TASK 6: Contextual Compression Retriever

In [46]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
documents = [
    Document(
        page_content="""
        Artificial Intelligence is used in healthcare for
        medical diagnosis and medical imaging.

        AI systems can analyze X-rays, MRI scans and CT scans
        to identify possible diseases.

        Artificial Intelligence is also used in finance for
        fraud detection and algorithmic trading.

        AI is also used in autonomous vehicles for navigation
        and object detection.
        """
    ),

    Document(
        page_content="""
        Machine Learning is a subset of Artificial Intelligence.

        Machine learning algorithms learn patterns from data
        and use those patterns to make predictions.

        Machine Learning is widely used in healthcare,
        finance and recommendation systems.
        """
    ),
]

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [ ]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [ ]:
query = "How is AI used in medical imaging?"

In [ ]:
normal_results = base_retriever.invoke(query)

In [ ]:

for i, doc in enumerate(normal_results, 1):

    print(f"\nDocument {i}:")
    print(doc.page_content)

In [ ]:
compressed_results = compression_retriever.invoke(query)


In [ ]:
for i, doc in enumerate(compressed_results, 1):

    print(f"\nDocument {i}:")
    print(doc.page_content)

# Part 4: YouTube Content RAG ChatBot (Mini Project)

## TASK 7:  Load Youtube Content

In [48]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [49]:
youtube_url = "https://www.youtube.com/watch?v=gfCg5srOjN0"

In [50]:
loader = YoutubeLoader.from_youtube_url(
    youtube_url,
    add_video_info=False
)

In [51]:
documents = loader.load()

print("Number of documents:", len(documents))

Number of documents: 1


In [52]:
print(documents[0].page_content[:3000])

There was a study from Harvard that showed that the men who lived the longest, they actually had 14 years [music] of extra healthy life if they just did five things. Don't overdrink, don't smoke, don't be overweight, don't stress, and eat healthy. If you just do the basics, that's already 14 extra healthy years. >> Wow. I'm a scientist, work at Harvard Medical School, have been there for 25 years, and my goal in life is to improve the life of everybody on the planet, potentially treat or cure all diseases. Here's a good point that I want to make for everybody. There are other things aging as well besides DNA damage. X-rays [music] break your DNA, by the way, but I can guarantee it's aging you every time we go to a rock concert [music] and we're blasting our ears. We go home with older ears than when we went to the concert. Tell me the ideal protocol for a young person. >> Try to eat less than three meals a day. See if you can join a gym so you can lift weights. If your gym has a sauna,

In [53]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [54]:
chunks = text_splitter.split_documents(documents)

In [55]:
len(chunks)

163

In [56]:
for i, chunk in enumerate(chunks[:3], 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)


--- Chunk 1 ---
There was a study from Harvard that showed that the men who lived the longest, they actually had 14 years [music] of extra healthy life if they just did five things. Don't overdrink, don't smoke, don't be overweight, don't stress, and eat healthy. If you just do the basics, that's already 14 extra healthy years. >> Wow. I'm a scientist, work at Harvard Medical School, have been there for 25 years, and my goal in life is to improve the life of everybody on the planet, potentially treat or cure all diseases. Here's a good point that I want to make for everybody. There are other things aging as well besides DNA damage. X-rays [music] break your DNA, by the way, but I can guarantee it's aging you every time we go to a rock concert [music] and we're blasting our ears. We go home with older ears than when we went to the concert. Tell me the ideal protocol for a young person. >> Try to eat less than three meals a day. See if you can join a gym so you can lift weights. If your

## TASK 8: Build Vector Store For Youtube Content

In [57]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [58]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)


In [59]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [60]:
query = "What is the main topic discussed in the video?"

In [61]:
results = retriever.invoke(query)

In [62]:
for i, doc in enumerate(results, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content)


Chunk 1:
explain from the basics. Let's say why do we age and how every body cell or every body organ is aging differently. >> I have a little structure here that I brought to show you this to explain it. So, this is a piece of DNA. Okay. And actually, I [music] have another model to show you guys this. I have a small favor to ask you. I need you to subscribe to our channel. The more subscribers we have, the better and bigger guests we can bring and provide you more value through these conversations. And the full audio experience of this show is also available on Spotify where you can follow us and listen to the new episodes as well. Now, let's get into the episode. If someone's watching you for the first time, okay, and they have no clue who you are and what you do. >> That's like most people watching this. >> No, no, of course. So many people know. My audience would definitely know you. Like a lot of them, right? And we're honored to have you here. >> Oh, thanks for having me on, Ra

## TASK 9: Build Youtube RAG Chatbot

In [67]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

In [65]:

chat_history = []

In [68]:

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a YouTube video assistant.
        Answer the user's question using ONLY the
        provided video transcript context.

        If the answer cannot be found in the transcript,
        say:
        "I couldn't find that information in the video."

        Do not make up information.

        Conversation history:
        {chat_history}

        Video context:
        {context}
        """
    ),
    (
        "human",
        "{question}"
    )
])

In [63]:
def ask_question(question):
    documents = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in documents
    )
    history = "\n".join(
        f"User: {message.content}"
        if isinstance(message, HumanMessage)
        else f"Assistant: {message.content}"
        for message in chat_history
    )

    messages = prompt.format_messages(chat_history=history,context=context,question=question)
    response = llm.invoke(messages)
    chat_history.append(HumanMessage(content=question))

    chat_history.append(AIMessage(content=response.content))

    return response.content


## TASK 10: Testing and Evaluation

In [ ]:
ask_question("What is the main topic of the video?")

'The main topic of the video is aging, specifically discussing why we age, how different body cells and organs age differently, and the scientific efforts to potentially treat or reverse aging.'

In [ ]:
ask_question("What does the speaker talk abouts")

In [ ]:
ask_question("Who is the person in the video")

In [ ]:
ask_question("Who is David Sinclair")

In [ ]:
ask_question("What are they taliking about in the video")

# Part 5: Observations and Insight

## TASK 11: Conceptual Questions

1. Difference between retriever-based RAG and normal prompting
- The LLM relies primarily on its training knowledge and the information included in the prompt.
- RAG retrieves external information and provides it to the LLM as context before generating the answer.
- 

2. Why vector stores are critical
- Vector stores allow documents to be represented as embeddings and searched using semantic similarity.

3. When to use MMR vs similarity search
- Similarity search is useful when you want the documents that are most semantically similar to the query.
- MMR is useful when the top results are highly redundant.

4. Benefits of Multi-Query Retrieval
- MultiQueryRetriever uses the LLM to generate different formulations of the original question.

5. Importance of Contextual Compression
- A retriever may find a relevant document but return much more information than the question requires.